In [ ]:
import sys
print("🔧 Instalando dependencias: pydantic")
!{sys.executable} -m pip install -q pydantic
print("✅ Instalación completada. Ejecuta las celdas siguientes.")

# AEM4L1 E05 — JSON del LLM → Pydantic

**Objetivo:** Validar que el JSON que devuelve un LLM coincida exactamente con nuestra estructura esperada.

**Contexto:** Un LLM (ChatGPT + Vision) extrae datos de una imagen. Pydantic actúa como **barrera de seguridad** entre el LLM y nuestra aplicación.

**Tipo:** Resuelto

**Uso:** listo para Google Colab. Ejecutá las celdas en orden.

## 📦 Dependencias

**Necesario instalar:**
- `pydantic`

**Cómo instalar:**
```bash
pip install pydantic
```

La celda de instalación está incluida a continuación. Ejecuta esa celda primero.

## ¿Por qué Pydantic aquí?

```
┌─────────────────────┐
│   Imagen           │
│  (recibo, cédula)  │
└──────────┬──────────┘
           │
           ▼
┌─────────────────────────┐
│   LLM (ChatGPT Vision) │
│  "Extrae los datos"     │
└──────────┬──────────────┘
           │
           ▼  JSON sin validar ⚠️
┌──────────────────────────┐
│  Pydantic BaseModel     │
│  "Valida estructura"    │ ◄── BARRERA DE SEGURIDAD
└──────────┬───────────────┘
           │
           ▼  JSON validado ✅
┌──────────────────────────┐
│  Tu aplicación          │
│  "Usa datos con seguridad" │
└──────────────────────────┘
```

Si el LLM comete errores (tipos incorrectos, campos faltantes, etc.), Pydantic lo detecta.

In [ ]:
import sys

print("=" * 60)
print("🔧 Instalando dependencias")
print("=" * 60)

# Instalar pydantic
print("\n📦 Instalando: pydantic")
!{sys.executable} -m pip install -q pydantic

print("✅ ¡Listo! Todas las dependencias instaladas.\n")

## Importaciones necesarias

En este notebook vamos a usar:
- **pydantic**: `BaseModel`, `ValidationError`, `field_validator`
- **typing**: `Optional` (campos opcionales)
- **datetime**: `date` (para parsear fechas automáticamente)
- **json**: Para simular JSON del LLM

In [ ]:
import json
from pydantic import BaseModel, ValidationError, field_validator
from datetime import date
from typing import Optional

print("=" * 60)
print("ESCENARIO: LLM extrae datos de un recibo")
print("=" * 60)

# Definimos la estructura esperada de un recibo
class ReceiptData(BaseModel):
    """
    Modelo para datos extraídos de un recibo.
    El LLM DEBE devolver EXACTAMENTE estos campos con estos tipos.
    """
    store_name: str          # Nombre de la tienda
    purchase_date: date      # Fecha (YYYY-MM-DD)
    total_amount: float      # Monto total en dinero
    payment_method: Optional[str] = None  # Método de pago (opcional)

print("\n✅ Caso 1: JSON VÁLIDO del LLM:")
print("-" * 60)

# Simulamos JSON que el LLM devuelve
valid_json = {
    "store_name": "Supermercado Norte",
    "purchase_date": "2026-06-12",  # String en formato ISO
    "total_amount": 8500.50,
    "payment_method": "tarjeta de crédito"
}

print(f"JSON recibido del LLM:")
print(json.dumps(valid_json, indent=2))

# Pydantic valida y convierte automáticamente
receipt = ReceiptData(**valid_json)
print(f"\n✅ Datos VALIDADOS por Pydantic:")
print(f"   {receipt}")
print(f"\n💡 Note que:")
print(f"   - Pydantic convirtió '2026-06-12' (str) → date")
print(f"   - purchase_date ahora es: {type(receipt.purchase_date)}")

print("\n\n❌ Caso 2: JSON INVÁLIDO del LLM:")
print("-" * 60)

# El LLM comete errores
invalid_json = {
    "store_name": "Supermercado Norte",
    "purchase_date": "12/06/2026",      # ❌ Formato fecha incorrecto
    "total_amount": "ocho mil quinientos"  # ❌ No es un número
}

print(f"JSON recibido del LLM:")
print(json.dumps(invalid_json, indent=2))

print(f"\nIntentando validar...")
try:
    receipt_invalid = ReceiptData(**invalid_json)
except ValidationError as e:
    print(f"\n❌ ¡Pydantic detectó ERRORES!")
    print(f"\nDetalles:")
    for error in e.errors():
        campo = error['loc'][0]
        tipo_error = error['type']
        msg = error['msg']
        print(f"\n   Campo: {campo}")
        print(f"   Tipo de error: {tipo_error}")
        print(f"   Mensaje: {msg}")

print("\n\n💡 Sin Pydantic:")
print("   Los datos inválidos se pasarían a la app")
print("   ❌ Errores solo aparecerían después en producción")
print("\n💡 CON Pydantic:")
print("   Los errores se detectan INMEDIATAMENTE")
print("   ✅ Sabemos exactamente qué está mal")

ModuleNotFoundError: No module named 'pydantic'